# Calibrate Benchmark (L0→L1)

Measures eager open_l0 load cost, read_stride sensitivity, write cost, full vs decimated L1 size.

In [ ]:
from pathlib import Path

from panoseti_analysis.io.bench import BenchResult, stage_timer, summarize
from panoseti_analysis.paths import REPO_ROOT

# ── Configure ───────────────────────────────────────────────────────────────────────────────
L0_STORE = Path("/path/to/store.dp_img16.module_1.L0.zarr")  # replace
OUT_BASE = Path("/tmp/calibrate_bench")

results: list[BenchResult] = []

## §1 Baseline: full L0 load + full-res L1 write

In [ ]:
import shutil

import xarray as xr

from panoseti_analysis.adapters.calibrate import run_calibrate

OUT_BASELINE = OUT_BASE / "baseline"
if OUT_BASELINE.exists():
    shutil.rmtree(OUT_BASELINE)

with stage_timer("calibrate_baseline", bytes_in=L0_STORE.stat().st_size) as r:
    run_calibrate(L0_STORE, OUT_BASELINE / "l1.zarr", checksum=False)

r.bytes_out = sum(f.stat().st_size for f in (OUT_BASELINE / "l1.zarr").rglob("*") if f.is_file())
results.append(r)
print(summarize(results))
print(
    f"Output frames: {xr.open_zarr(str(OUT_BASELINE / 'l1.zarr'), consolidated=False).sizes['time']}"
)

## §2 Decimated: read_stride=10

In [ ]:
import shutil

from panoseti_analysis.adapters.calibrate import run_calibrate

OUT_STRIDED = OUT_BASE / "strided"
if OUT_STRIDED.exists():
    shutil.rmtree(OUT_STRIDED)

with stage_timer("calibrate_read_stride_10", bytes_in=L0_STORE.stat().st_size) as r:
    run_calibrate(L0_STORE, OUT_STRIDED / "l1.zarr", read_stride=10, checksum=False)

r.bytes_out = sum(f.stat().st_size for f in (OUT_STRIDED / "l1.zarr").rglob("*") if f.is_file())
results.append(r)
print(summarize(results))
print(
    f"Output frames: {xr.open_zarr(str(OUT_STRIDED / 'l1.zarr'), consolidated=False).sizes['time']}"
)

## §3 Summary

In [ ]:
print(summarize(results))